In [1]:
from datasets import load_dataset
def load_wildguard():
    wildguard_test = load_dataset("allenai/wildguardmix", "wildguardtest")["test"]
    wildguard_test = wildguard_test.to_pandas()
    wildguard_test = wildguard_test.dropna()

    prompts = wildguard_test["prompt"].values.tolist()
    labels = wildguard_test["prompt_harm_label"].values.tolist()
    map2id = {"unharmful": 0, "harmful": 1}
    labels= [map2id[x] for x in labels]
    return prompts,labels,wildguard_test

In [2]:
real_prompt, real_labels,wild_df = load_wildguard()

Using the latest cached version of the dataset since allenai/wildguardmix couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wildguardtest' at /home/unica/.cache/huggingface/datasets/allenai___wildguardmix/wildguardtest/0.0.0/d29c47f41c8b51348b5c8e8c81c039b3132b66d1 (last modified on Mon May 26 10:32:15 2025).


In [3]:
import pandas as pd
import os
from glob import glob
from sklearn.metrics import accuracy_score, f1_score

def accuracy_f1(real, preds):
    return {
        "ACC": round(accuracy_score(real, preds), 4),
        "F1": round(f1_score(real, preds), 4)
    }

def wildguard_scores(pred_df):
    wild_adv=wild_df[wild_df["adversarial"]==True]    
    pred_adv=pred_df[pred_df["text"].isin(wild_adv["prompt"])]    
    wild_van=wild_df[wild_df["adversarial"]==False]
    pred_van=pred_df[pred_df["text"].isin(wild_van["prompt"])]
    
    overall=accuracy_f1(pred_df["real"].values.tolist(),pred_df["pred"].values.tolist())
    vanilla=accuracy_f1(pred_van["real"].values.tolist(),pred_van["pred"].values.tolist())
    adversarial=accuracy_f1(pred_adv["real"].values.tolist(),pred_adv["pred"].values.tolist())
    # Restituiamo una lista di dict per facilitare il DataFrame
    return [
        {"Method": "Overall", **overall},
        {"Method": "Vanilla", **vanilla},
        {"Method": "Adversarial", **adversarial}
    ]

def compute_scores(base_path,models=None):
    # Path base (usiamo fold-0 solo per listare i modelli disponibili)
    if not models:
        models = [x for x in os.listdir(base_path) if not x.startswith(".")]
    fold=0
    for model in models:
        all_folds_results = []
        all_folds_wild = []

       
        preds_files = glob(f"{base_path}/{model}/*.json")
            
        for f_path in preds_files:
            dataset_name = os.path.basename(f_path).replace(".json", "")
            pred_df = pd.read_json(f_path)
            
            if dataset_name != "WildGuard":
                # IN-DOMAIN / STANDARD OOD
                metrics = accuracy_f1(pred_df["real"], pred_df["pred"])
                metrics.update({"dataset": dataset_name, "fold": fold})
                all_folds_results.append(metrics)
            else:
                # WILDGUARD (OUT-OF-DOMAIN con split interni)
                metrics_list = wildguard_scores(pred_df)
                for m in metrics_list:
                    m.update({"dataset": "WildGuard", "fold": fold})
                    all_folds_wild.append(m)

        print(f"\n--- REPORT FOR MODEL: {model} ---")
        
        # 1. Processing IN-DOMAIN (o altri dataset standard)
        if all_folds_results:
            df_in = pd.DataFrame(all_folds_results)
            # Raggruppiamo per dataset e calcoliamo media e std
            in_summary = df_in.groupby("dataset")[["ACC", "F1"]].agg(["mean", "std"]).round(3)
            print("\n[IN-DOMAIN / STANDARD DATASETS]")
            print(in_summary)

        # 2. Processing WILDGUARD (OOD con split)
        if all_folds_wild:
            df_wild = pd.DataFrame(all_folds_wild)
            # Raggruppiamo per il tipo di split (Method)
            wild_summary = df_wild.groupby("Method")[["ACC", "F1"]].agg(["mean", "std"]).round(3)
            print("\n[OUT-OF-DOMAIN: WILDGUARD]")
            print(wild_summary)
            
        print("-" * 50)



### WITHOUT HARMLESS

In [4]:
folder="FT"
base_path = f"./output/fold-0/parsed/{folder}/abl_harmless"
compute_scores(base_path,models=["llama3.2-3","mistral","gemma2"])


--- REPORT FOR MODEL: llama3.2-3 ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.802 NaN  0.829 NaN
OrBench    0.735 NaN  0.000 NaN
Remedy     0.951 NaN  0.951 NaN
ToxicChat  0.960 NaN  0.753 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           
Adversarial  0.751 NaN  0.681 NaN
Overall      0.838 NaN  0.804 NaN
Vanilla      0.913 NaN  0.900 NaN
--------------------------------------------------

--- REPORT FOR MODEL: mistral ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.799 NaN  0.827 NaN
OrBench    0.691 NaN  0.000 NaN
Remedy     0.958 NaN  0.957 NaN
ToxicChat  0.962 NaN  0.767 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           


### ONLY LABELS

In [5]:
folder="FT"
base_path = f"./output/fold-0/parsed/{folder}/no_rationale"
compute_scores(base_path,models=["llama3.2-3","mistral","gemma2"])


--- REPORT FOR MODEL: llama3.2-3 ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.811 NaN  0.840 NaN
OrBench    0.682 NaN  0.000 NaN
Remedy     0.951 NaN  0.951 NaN
ToxicChat  0.953 NaN  0.738 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           
Adversarial  0.772 NaN  0.741 NaN
Overall      0.853 NaN  0.834 NaN
Vanilla      0.924 NaN  0.914 NaN
--------------------------------------------------

--- REPORT FOR MODEL: mistral ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.838 NaN  0.866 NaN
OrBench    0.664 NaN  0.000 NaN
Remedy     0.963 NaN  0.963 NaN
ToxicChat  0.953 NaN  0.736 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           


## FULL

In [6]:
base_path = f"../output/fold-0/parsed/{folder}/"
compute_scores(base_path,models=["llama3.2-3","mistral","gemma2"])


--- REPORT FOR MODEL: llama3.2-3 ---
--------------------------------------------------

--- REPORT FOR MODEL: mistral ---
--------------------------------------------------

--- REPORT FOR MODEL: gemma2 ---

[IN-DOMAIN / STANDARD DATASETS]
           ACC         F1    
          mean std   mean std
dataset                      
Remedy   0.953 NaN  0.955 NaN
--------------------------------------------------
